# My run crashed — where do I find info?

## Three quick ways to tell if your run crashed

"No longer listed in `qstat`" isn't proof of success — check one of these instead:

1. **Archive directory** — did new history files show up under
   `/glade/derecho/scratch/$USER/archive/$CASE/`? If it's still empty (or missing the month
   you expected), the run likely didn't finish.
2. **Restart files** — are there `*.r.*` and `rpointer.*` files in the run directory (or
   under `rest/` once archived) with a timestamp matching your run's expected end date? Their
   absence means the model never reached a clean stopping point.
3. **`CASE/CaseStatus`** — does it end with `case.run success` and `st_archive success`? See
   ["What does success actually look like?"](../04.CESM-Workflow-Quickstart/four_commands.ipynb)
   for a worked example, and the failure-side example in the checklist below.

If your run crashed, examine the **log files** in the run directory, `$RUNDIR`. This is
the first place to look.

- While the run is in progress, log files live in `$RUNDIR`.
- Once the run completes (successfully), they are moved to `CASE/logs`.

A crashed run typically won't move its logs to `CASE/logs`, so check `$RUNDIR` directly, e.g.:
```bash
cd /glade/derecho/scratch/$USER/$CASE/run
ls -ltr *log*
```
This lists log files oldest-to-newest, so the component that crashed — the one whose log
stopped updating — is usually near the bottom.

Search for `ERROR` or `ABORT` rather than just reading the last few lines:
```bash
grep -in "error\|abort" cesm.log.* | tail -20
```

<div class="alert alert-warning">
The <strong>very bottom</strong> of a log file is often just the batch system's normal
shutdown message, not the actual cause — that can make a crash look like nothing went wrong.
The real error is usually a little <em>earlier</em> in the file, right around the last
timestep the model successfully completed.
</div>

## General troubleshooting checklist

1. Check `qstat -u $USER` — did the job even start, or is it still queued?
2. Run through the three checks above. A failed `CaseStatus` shows `error` instead of
   `success`:
   ```text
   2026-09-02 14:32:09: case.run starting
   ---------------------------------------------------
   2026-09-02 14:33:41: case.run error
   ```
   The timestamp of the `error` line tells you roughly when in the run it died, and which
   step (`case.setup`/`case.build`/`case.run`) to focus on.
3. If the **build** failed, check the build logs under `$CASEROOT/bld/*.bldlog.*`.
4. If the **run** failed, work through the log files as described above, starting with
   `cesm.log.*`.
5. Re-read the [xml files](../05.Run-length/xml_files.ipynb) section — many crashes come from
   an unrealistic `STOP_N`/`RESUBMIT` combination that runs out of allocated wallclock time.
